In [ ]:
%pip install pandas yfinance

In [ ]:
import pandas as pd
import yfinance as yf

In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/Project/dataset/marco_and_industry/'

In [ ]:
TICKERS = [
    "2330.TW", "2317.TW", "2454.TW", "2308.TW", "2382.TW", "2881.TW", "2891.TW", "2882.TW", "2303.TW",
    "2412.TW", "2884.TW", "2886.TW", "3711.TW", "2357.TW", "1216.TW", "2885.TW", "2345.TW", "3231.TW",
    "3034.TW", "2892.TW", "2379.TW", "6669.TW", "2890.TW", "5880.TW", "2880.TW", "2383.TW", "3661.TW",
    "3017.TW", "2883.TW", "3008.TW"
]


In [ ]:
TW_PROXIES = {
    # 半導體/電子：全球半導體指數ETF（美股），反映整體景氣循環
    "SOXX_Semiconductor_ETF": "SOXX",
    "SMH_Semiconductor_ETF": "SMH",
    # 原物料與成本：銅、油價、鐵礦砂（對電子、鋼鐵、製造成本敏感）
    "Copper_Futures": "HG=F",      # COMEX Copper
    "WTI_Crude_Oil": "CL=F",       # WTI Crude
    "Brent_Crude_Oil": "BZ=F",     # Brent Crude
    "Iron_Ore_62pct": "TIO=F",     # SGX Iron Ore 62%
    # 航運：乾散貨、集裝箱 proxy
    "Dry_Bulk_Shipping_ETF": "BDRY",  # 乾散貨運價代理
    "Global_Shipping_ETF": "SEA",     # 全球航運公司ETF
    # 鋼鐵產業：鋼鐵ETF
    "Global_Steel_ETF": "SLX",
    # 匯率與美元：台幣、美元指數（影響外資與出口競爭力）
    "USD_TWD": "TWD=X",
    "US_Dollar_Index": "DX-Y.NYB",
    # 參考大盤（台股/費半）：
    "TAIEX": "^TWII",
    "PHLX_Semiconductor_Index": "^SOX"
}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def fetch_yahoo_proxies(mapping: dict, start="2020-01-01"):
    prices = {}
    for name, ticker in mapping.items():
        try:
            df = yf.download(
                ticker,
                start=start,
                progress=False,
                auto_adjust=True,
                actions=False
            )
            if df is None or df.empty:
                print(f"⚠ {name} ({ticker}) 無資料，已略過")
                continue
            s = _extract_close_series(df, ticker, name)
            prices[name] = s
        except Exception as e1:
            # fallback：改走 history()
            try:
                df = yf.Ticker(ticker).history(start=start, auto_adjust=True)
                if df is None or df.empty:
                    print(f"⚠ {name} ({ticker}) 無資料，已略過")
                    continue
                s = _extract_close_series(df, ticker, name)
                prices[name] = s
            except Exception as e2:
                print(f"⚠ 下載 {name} ({ticker}) 失敗：{e2}")

    if not prices:
        return pd.DataFrame()

    out = pd.concat(prices.values(), axis=1).sort_index()
    out.index.name = "Date"
    return out.ffill()

def _extract_close_series(df: pd.DataFrame, ticker: str, out_name: str) -> pd.Series:
    """從 yfinance 的 DataFrame 中穩定取出 Close 並回傳 Series（處理單層/多層欄位）"""
    if df is None or df.empty:
        raise ValueError("empty dataframe")

    # 單層欄位
    if not isinstance(df.columns, pd.MultiIndex):
        col = "Adj Close" if "Adj Close" in df.columns else "Close"
        if col not in df.columns:
            raise KeyError(f"{col} not found")
        return df[col].rename(out_name)

    lvl0 = df.columns.get_level_values(0)
    if "Close" in lvl0:
        s = df.xs("Close", axis=1, level=0)
    else:
        lvl1 = df.columns.get_level_values(1)
        if "Close" in lvl1:
            s = df.xs("Close", axis=1, level=1)
        else:
            raise KeyError("Close not found in MultiIndex columns")

    if isinstance(s, pd.DataFrame):
        if ticker in s.columns:
            s = s[ticker]
        else:
            s = s.iloc[:, 0]
    return s.rename(out_name)

In [ ]:
tw_df = fetch_yahoo_proxies(TW_PROXIES, start="2020-01-01")

In [ ]:
tw_df

,SOXX_Semiconductor_ETF,SMH_Semiconductor_ETF,Copper_Futures,WTI_Crude_Oil,Brent_Crude_Oil,Iron_Ore_62pct,Dry_Bulk_Shipping_ETF,Global_Shipping_ETF,Global_Steel_ETF,USD_TWD,US_Dollar_Index,TAIEX,PHLX_Semiconductor_Index
Date,,,,,,,,,,,,,
2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.898001,NaN,NaN,NaN
2020-01-02,81.308868,69.920441,2.8330,61.180000,66.250000,92.580002,13.961,NaN,31.094810,29.844000,96.849998,12100.480469,1887.910034
2020-01-03,79.792381,68.775078,2.7985,63.049999,68.599998,93.410004,13.050,NaN,30.661240,29.900000,96.839996,12110.429688,1853.979980
2020-01-06,78.961166,68.040482,2.8005,63.270000,68.910004,93.849998,11.920,NaN,30.366732,30.035000,96.669998,11953.360352,1834.680054
2020-01-07,80.414177,69.176193,2.8040,62.700001,68.269997,93.730003,11.895,NaN,30.579430,30.047001,96.980003,11880.320312,1867.280029
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-05,238.729996,287.100006,4.3640,65.160004,67.639999,101.360001,7.770,14.460,67.370003,29.884100,98.779999,23660.589844,5561.689941
2025-08-06,236.929993,286.619995,4.3910,64.349998,66.889999,100.919998,7.730,14.555,66.919998,29.937401,98.180000,23447.359375,5550.299805
2025-08-07,240.839996,291.119995,4.3785,63.880001,66.430000,101.209999,7.750,14.618,67.610001,29.914700,98.400002,24003.769531,5633.700195


In [ ]:
import numpy as np
def add_change_features(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    num_cols = df_copy.select_dtypes(include=["number"]).columns
    new_features = pd.DataFrame(index=df_copy.index)

    for col in num_cols:
        series_no_na = df_copy[col].dropna()

        # 檢查 series 是否有足夠的資料點來計算變化率
        if len(series_no_na) > 1:
            # 計算百分比變化率
            pct_chg = series_no_na.pct_change()
            new_features[f"{col}_pct_chg"] = pct_chg.replace([np.inf, -np.inf], np.nan)

            # 計算對數差異
            with np.errstate(divide="ignore", invalid="ignore"):
                logdiff = np.log(series_no_na).diff()
            new_features[f"{col}_logdiff"] = logdiff.replace([np.inf, -np.inf], np.nan)

            # 計算絕對差異
            diff = series_no_na.diff()
            new_features[f"{col}_diff"] = diff

    # 將原始 DataFrame 與新特徵合併
    return pd.concat([df_copy, new_features], axis=1)

In [ ]:
tw_df_copy = tw_df.copy()

In [ ]:
tw_df_with_changes = add_change_features(tw_df_copy)

In [ ]:
tw_df_with_changes

,SOXX_Semiconductor_ETF,SMH_Semiconductor_ETF,Copper_Futures,WTI_Crude_Oil,Brent_Crude_Oil,Iron_Ore_62pct,Dry_Bulk_Shipping_ETF,Global_Shipping_ETF,Global_Steel_ETF,USD_TWD,...,USD_TWD_diff,US_Dollar_Index_pct_chg,US_Dollar_Index_logdiff,US_Dollar_Index_diff,TAIEX_pct_chg,TAIEX_logdiff,TAIEX_diff,PHLX_Semiconductor_Index_pct_chg,PHLX_Semiconductor_Index_logdiff,PHLX_Semiconductor_Index_diff
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.898001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02,81.308868,69.920441,2.8330,61.180000,66.250000,92.580002,13.961,NaN,31.094810,29.844000,...,-0.054001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,79.792381,68.775078,2.7985,63.049999,68.599998,93.410004,13.050,NaN,30.661240,29.900000,...,0.056000,-0.000103,-0.000103,-0.010002,0.000822,0.000822,9.949219,-0.017972,-0.018136,-33.930054
2020-01-06,78.961166,68.040482,2.8005,63.270000,68.910004,93.849998,11.920,NaN,30.366732,30.035000,...,0.135000,-0.001755,-0.001757,-0.169998,-0.012970,-0.013055,-157.069336,-0.010410,-0.010465,-19.299927
2020-01-07,80.414177,69.176193,2.8040,62.700001,68.269997,93.730003,11.895,NaN,30.579430,30.047001,...,0.012001,0.003207,0.003202,0.310005,-0.006110,-0.006129,-73.040039,0.017769,0.017613,32.599976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-05,238.729996,287.100006,4.3640,65.160004,67.639999,101.360001,7.770,14.460,67.370003,29.884100,...,0.069099,0.000000,0.000000,0.000000,0.012047,0.011975,281.650391,-0.011151,-0.011214,-62.720215
2025-08-06,236.929993,286.619995,4.3910,64.349998,66.889999,100.919998,7.730,14.555,66.919998,29.937401,...,0.053301,-0.006074,-0.006093,-0.599998,-0.009012,-0.009053,-213.230469,-0.002048,-0.002050,-11.390137
2025-08-07,240.839996,291.119995,4.3785,63.880001,66.430000,101.209999,7.750,14.618,67.610001,29.914700,...,-0.022701,0.002241,0.002238,0.220001,0.023730,0.023453,556.410156,0.015026,0.014915,83.400391


In [ ]:
tw_df_with_changes_filled = tw_df_with_changes.ffill().bfill()
tw_df_with_changes_filled

,SOXX_Semiconductor_ETF,SMH_Semiconductor_ETF,Copper_Futures,WTI_Crude_Oil,Brent_Crude_Oil,Iron_Ore_62pct,Dry_Bulk_Shipping_ETF,Global_Shipping_ETF,Global_Steel_ETF,USD_TWD,...,USD_TWD_diff,US_Dollar_Index_pct_chg,US_Dollar_Index_logdiff,US_Dollar_Index_diff,TAIEX_pct_chg,TAIEX_logdiff,TAIEX_diff,PHLX_Semiconductor_Index_pct_chg,PHLX_Semiconductor_Index_logdiff,PHLX_Semiconductor_Index_diff
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-01,81.308868,69.920441,2.8330,61.180000,66.250000,92.580002,13.961,13.015002,31.094810,29.898001,...,-0.054001,-0.000103,-0.000103,-0.010002,0.000822,0.000822,9.949219,-0.017972,-0.018136,-33.930054
2020-01-02,81.308868,69.920441,2.8330,61.180000,66.250000,92.580002,13.961,13.015002,31.094810,29.844000,...,-0.054001,-0.000103,-0.000103,-0.010002,0.000822,0.000822,9.949219,-0.017972,-0.018136,-33.930054
2020-01-03,79.792381,68.775078,2.7985,63.049999,68.599998,93.410004,13.050,13.015002,30.661240,29.900000,...,0.056000,-0.000103,-0.000103,-0.010002,0.000822,0.000822,9.949219,-0.017972,-0.018136,-33.930054
2020-01-06,78.961166,68.040482,2.8005,63.270000,68.910004,93.849998,11.920,13.015002,30.366732,30.035000,...,0.135000,-0.001755,-0.001757,-0.169998,-0.012970,-0.013055,-157.069336,-0.010410,-0.010465,-19.299927
2020-01-07,80.414177,69.176193,2.8040,62.700001,68.269997,93.730003,11.895,13.015002,30.579430,30.047001,...,0.012001,0.003207,0.003202,0.310005,-0.006110,-0.006129,-73.040039,0.017769,0.017613,32.599976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-05,238.729996,287.100006,4.3640,65.160004,67.639999,101.360001,7.770,14.460000,67.370003,29.884100,...,0.069099,0.000000,0.000000,0.000000,0.012047,0.011975,281.650391,-0.011151,-0.011214,-62.720215
2025-08-06,236.929993,286.619995,4.3910,64.349998,66.889999,100.919998,7.730,14.555000,66.919998,29.937401,...,0.053301,-0.006074,-0.006093,-0.599998,-0.009012,-0.009053,-213.230469,-0.002048,-0.002050,-11.390137
2025-08-07,240.839996,291.119995,4.3785,63.880001,66.430000,101.209999,7.750,14.618000,67.610001,29.914700,...,-0.022701,0.002241,0.002238,0.220001,0.023730,0.023453,556.410156,0.015026,0.014915,83.400391


In [ ]:
tw_df_with_changes_filled.index = pd.to_datetime(tw_df_with_changes_filled.index)
tw_df_with_changes_filled = tw_df_with_changes_filled[tw_df_with_changes_filled.index >= "2020-01-01"]
tw_df_with_changes_filled.filter(regex=r"(_pct_chg|_logdiff|_diff)$").to_csv(f"{DRIVE_PATH}/tw_industry_related_dataset.csv")